# Long-read transcript reconstruction benchmark

This notebook reads the compact bundle created by `extract_pairwise_benchmark.py`. It is a post-workflow analysis: plots describe model production and pairwise concordance, not truth-based recall.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BUNDLE = Path(os.environ.get('PAIRWISE_BUNDLE', '.')).resolve()
PLOTS = BUNDLE / 'plots'
PLOTS.mkdir(exist_ok=True)
print(BUNDLE)

In [ ]:
pair_summary = pd.read_csv(BUNDLE / 'pair_summary.tsv', sep='\t')
tool_summary = pd.read_csv(BUNDLE / 'tool_technology_summary.tsv', sep='\t')
model_inventory = pd.read_csv(BUNDLE / 'model_inventory_summary.tsv', sep='\t')
whole_models = pd.read_csv(BUNDLE / 'whole_model_class_counts.tsv', sep='\t')
status_path = BUNDLE / 'metric_status_summary.tsv'
metric_status = pd.read_csv(status_path, sep='\t') if status_path.exists() else pd.DataFrame()
for frame in (model_inventory, whole_models):
    for column in ('n_models', 'n_loci', 'n_exons', 'n_models_with_evidence'):
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors='coerce').fillna(0)
display(pair_summary.head())
display(tool_summary)

## Model production

Model and locus counts should be interpreted together; a larger model set is not automatically better.

In [ ]:
plot_data = (model_inventory.groupby(['tool', 'technology'], dropna=False, as_index=False)
             .agg(n_models=('n_models', 'sum'), n_loci=('n_loci', 'sum')))
ax = plot_data.set_index('tool')[['n_models', 'n_loci']].plot(kind='bar', figsize=(10, 5), logy=True)
ax.set_ylabel('Count (log scale)')
ax.set_title('Model and locus production')
plt.tight_layout()
plt.savefig(PLOTS / 'model_and_locus_counts.png', dpi=180)
plt.show()

## Structural concordance

These are pairwise agreement summaries. They are not recall or precision.

In [ ]:
metrics = ['intron_chain_concordance', 'end_concordance', 'utr_concordance', 'whole_model_concordance']
available = [metric for metric in metrics if metric in tool_summary.columns]
long = tool_summary.melt(id_vars=['tool_a', 'tool_b', 'technology_a'], value_vars=available,
                         var_name='metric', value_name='concordance')
long['concordance'] = pd.to_numeric(long['concordance'], errors='coerce')
long['comparison'] = long['tool_a'] + ' vs ' + long['tool_b']
ax = long.pivot_table(index='comparison', columns='metric', values='concordance').plot(kind='bar', figsize=(12, 6))
ax.set_ylim(0, 1)
ax.set_ylabel('Pairwise concordance')
ax.set_title('Structural concordance by tool comparison')
plt.tight_layout()
plt.savefig(PLOTS / 'structural_concordance.png', dpi=180)
plt.show()

## Whole-model classes and metric status

In [ ]:
class_plot = (whole_models.groupby(['whole_model_class'], as_index=False)['n_models'].sum()
              .sort_values('n_models', ascending=False))
class_plot.plot(kind='bar', x='whole_model_class', y='n_models', legend=False, figsize=(10, 5))
plt.ylabel('Comparisons')
plt.title('Whole-model comparison classes')
plt.tight_layout()
plt.savefig(PLOTS / 'whole_model_classes.png', dpi=180)
plt.show()

if not metric_status.empty:
    display(metric_status.pivot_table(index='metric', columns='status', values='n_pairs', aggfunc='sum', fill_value=0))